In [1]:
import requests
from requests.packages.urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
import csv
from tqdm import tqdm

In [2]:
class TimeoutHttpAdapter(HTTPAdapter):
    def __init__(self, timeout=None, *args, **kwargs):
        self.timeout = timeout
        if "timeout" in kwargs:
            del kwargs["timeout"]
        super().__init__(*args, **kwargs)

    def send(self, *args, **kwargs):
        kwargs['timeout'] = self.timeout
        return super().send(*args, **kwargs)

In [14]:
r=requests.Session()
retry_strategy = Retry(
            total=5,
            status_forcelist=[104, 429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET" "POST", "PUT", "DELETE", "OPTIONS", "TRACE"],
            backoff_factor=2
        )
r.headers["User-Agent"]='My User Agent 2.0'
r.mount('https://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))
r.mount('http://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))

In [48]:
scrapped_doc=[]
for page in range(2,12):
    link='https://www.mondaq.com/5/India/Criminal-Law/?tab=morenews&pageNumber='+str(page)+'&order=0'
    req = r.get(link)
    soup = BeautifulSoup(req.content, 'html.parser')
    s = soup.find_all('tr', class_='results')
    link_list=[]
    for li in s:
        a = li.find("a")
        if a:
            link_list.append(a.attrs["href"])
    for i in range(len(link_list)):
        link_list[i]='https://www.mondaq.com'+link_list[i]
    for i in range(len(link_list)):
        req=r.get(link_list[i])
        soup = BeautifulSoup(req.content.decode(), "html.parser")
        title=soup.find('div',class_='article-title').h1.text
        content=soup.find('div',class_='article-body').text
        title=" ".join(title.split())
        scrapped_doc.append([title,content])

In [50]:
fields = ['title', 'content'] 
with open('legal_articles_for_crime_mondaq.csv', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(scrapped_doc)

In [51]:
f.close()